# 09 - 流式处理与回调（Streaming & Callbacks）

## 学习目标
- 掌握 .stream() 同步流式处理和 .astream() 异步流式处理
- 使用 .astream_events() 获取事件级别的流式输出
- 实现自定义 BaseCallbackHandler 监控链的执行
- 构建 Token 计数和成本追踪回调
- 配置 LangSmith 追踪
- 在链中通过 config 传递回调

In [ ]:
# 安装必要依赖（如未安装请取消注释）
# !pip install langchain langchain-openai langchain-community

## 1. .stream() - 同步流式处理

stream() 方法返回一个迭代器，每次产出链的一个中间结果块（chunk）。
适用于 LLM 输出的逐字展示、进度监控等场景。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("=== .stream() 同步流式处理 ===\n")

# 构建一个简单的 LCEL 链
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手，请简洁回答。"),
    ("human", "{question}"),
])

# 注意: 以下代码需要 OpenAI API key
print("# 模拟流式输出示例（实际需要 LLM）:\n")

print("# === 非流式调用 ===")
print("# 使用 .invoke()：等待完整结果")
print("result = chain.invoke({'question': '解释量子计算'})")
print("# 结果一次性返回，用户需要等待完整生成时间")
print()

print("# === 流式调用 ===")
print("# 使用 .stream()：逐个 chunk 产出")
print("""
for chunk in chain.stream({'question': '解释量子计算'}):
    print(chunk, end='', flush=True)  # 逐字打印
""")
print()
print("# stream() 返回的每个 chunk 是模型生成的一个 token/词")
print("# 用户体验: 看到文字逐字出现，而非等待后一次性显示")

print("\n" + "=" * 60)

print("# === 使用 RunnablePassthrough 监控流式输出 ===")
print("""
from langchain_core.runnables import RunnableLambda

def log_chunk(chunk):
    '''记录每个 chunk'''
    print(f"[CHUNK] {repr(chunk)}")
    return chunk

# 在链中插入日志步骤
chain_with_logging = chain | RunnableLambda(log_chunk)

for chunk in chain_with_logging.stream({'question': '你好'}):
    pass  # 每个 chunk 都会被 log_chunk 处理
""")

print("\n# === 模拟流式输出演示 ===")
import time
import sys

# 模拟 LLM 的流式输出
simulated_response = "量子计算是利用量子力学原理进行信息处理的计算方式。"

print("模拟流式输出效果:")
for char in simulated_response:
    print(char, end="", flush=True)
    time.sleep(0.05)  # 模拟生成延迟
print()

print("\n# .stream() 的实用场景:")
print("# 1. 聊天界面逐字显示回答")
print("# 2. 实时显示生成进度")
print("# 3. 提前取消生成（break 退出循环）")
print("# 4. 对每个 chunk 做后处理（如敏感词过滤）")
print("# 5. 记录生成过程中的中间状态")

## 2. .astream() - 异步流式处理

astream() 是 stream() 的异步版本，适用于异步 Web 框架（FastAPI 等）。
使用 async for 迭代异步生成器。

In [ ]:
import asyncio

print("=== .astream() 异步流式处理 ===\n")

print("# 异步流式处理的基本模式:")
print("""
import asyncio

async def async_stream_example(chain, query):
    '''异步流式处理示例'''
    full_response = []
    
    async for chunk in chain.astream({'question': query}):
        # 处理每个 chunk
        full_response.append(chunk)
        # 可以在这里发送 SSE (Server-Sent Events)
        yield chunk
    
    return ''.join(full_response)

# 在 FastAPI 中使用:
# @app.post('/chat/stream')
# async def chat_stream(request: ChatRequest):
#     async def generate():
#         async for chunk in chain.astream({'question': request.message}):
#             yield f"data: {chunk}\\n\\n"
#     return StreamingResponse(generate(), media_type='text/event-stream')
""")

print("\n" + "=" * 60)

print("# === 模拟异步流式输出 ===")

async def simulate_async_stream(text: str, delay: float = 0.1):
    """模拟异步流式输出"""
    for char in text:
        await asyncio.sleep(delay)
        yield char

async def demo_async_stream():
    """演示异步流式处理"""
    print("开始异步流式输出:")
    chunks = []
    async for chunk in simulate_async_stream("异步流式处理允许在等待 I/O 时处理其他任务。"):
        chunks.append(chunk)
        print(chunk, end="", flush=True)
    print(f"\n总共接收 {len(chunks)} 个chunks")
    return "".join(chunks)

# 运行异步演示
asyncio.run(demo_async_stream())

print("\n# .astream() vs .stream():")
print("# .stream():   同步迭代器，阻塞当前线程")
print("# .astream():  异步迭代器，不阻塞事件循环")
print("# 选择: Web服务用 .astream()，脚本用 .stream()")
print()
print("# 异步并发处理多个请求:")
print("""
async def process_multiple_queries(chain, queries):
    tasks = [chain.ainvoke(q) for q in queries]
    results = await asyncio.gather(*tasks)
    return results
""")

## 3. .astream_events() - 事件级别的流式处理

astream_events() 是最细粒度的流式接口，返回链中每个组件的生命周期事件。
可以精确知道"模型开始思考"、"工具被调用"、"链完成"等事件。

In [ ]:
print("=== .astream_events() 事件级流式处理 ===\n")

print("# astream_events() 返回的事件类型:")

events_table = [
    ("on_chain_start", "链开始执行", "chain"),
    ("on_chain_end", "链执行完成（携带输出）", "chain"),
    ("on_chat_model_start", "LLM 开始生成", "chat_model"),
    ("on_chat_model_stream", "LLM 流式输出每个 token", "chat_model"),
    ("on_chat_model_end", "LLM 生成完成", "chat_model"),
    ("on_retriever_start", "检索器开始检索", "retriever"),
    ("on_retriever_end", "检索器完成（携带文档）", "retriever"),
    ("on_tool_start", "工具开始执行", "tool"),
    ("on_tool_end", "工具执行完成（携带结果）", "tool"),
    ("on_tool_error", "工具执行出错", "tool"),
]

for event, desc, component in events_table:
    print(f"  {event:<24} | {desc:<32} | 来源: {component}")

print("\n" + "=" * 60)

print("# 基本使用模式:")
print("""
async def stream_with_events(chain, query):
    '''按事件类型过滤流式输出'''
    
    async for event in chain.astream_events(
        {'question': query},
        version="v2",  # 使用 v2 版本（推荐）
        config={"tags": ["streaming-demo"]},
    ):
        kind = event["event"]
        name = event.get("name", "")
        
        # 只关注流式输出事件
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(content, end="", flush=True)
        
        # 关注检索事件
        elif kind == "on_retriever_end":
            docs = event["data"]["output"]
            print(f"\\n[检索到 {len(docs)} 个文档]")
        
        # 关注工具调用
        elif kind == "on_tool_start":
            print(f"\\n[调用工具: {name}]")
        elif kind == "on_tool_end":
            print(f"[工具 {name} 完成]")
""")

print("\n# === 按标签/名称/事件类型过滤 ===")
print("""
# 只监听特定事件类型
async for event in chain.astream_events(
    input,
    version="v2",
    include_types=["chat_model", "retriever"],  # 只包含这些类型
    exclude_tags=["internal"],                    # 排除某些标签
    include_names=["final_llm", "doc_retriever"], # 只包含特定名称
):
    ...
""")

print("\n# 典型应用场景:")
print("# 1. 聊天界面: 显示'正在思考...' -> '正在检索文档...' -> 流式输出")
print("# 2. 调试: 精确追踪链中每个步骤的输入输出")
print("# 3. 性能监控: 记录每个步骤的耗时")
print("# 4. 多步骤展示: 显示 AI 的思考过程和处理步骤")
print("# 5. 安全审计: 记录所有工具调用和检索操作")

## 4. 自定义 BaseCallbackHandler - 回调处理器

通过继承 BaseCallbackHandler 实现自定义的事件监听和处理逻辑。
回调可以记录日志、计数 token、追踪成本、触发告警等。

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from typing import Any, Dict, List
from uuid import UUID
import time


class DebugLoggingCallback(BaseCallbackHandler):
    """
    完整的调试日志回调处理器。
    
    记录链中所有关键事件: LLM调用、检索、工具执行、错误等。
    """
    
    def __init__(self):
        super().__init__()
        self._start_times: Dict[str, float] = {}
        self._chain_stack: List[str] = []
        self.llm_call_count = 0
        self.tool_call_count = 0
        self.retriever_call_count = 0
    
    # === LLM 回调 ===
    
    def on_llm_start(
        self,
        serialized: Dict[str, Any],
        prompts: List[str],
        **kwargs: Any,
    ) -> None:
        """LLM 开始生成时调用"""
        self.llm_call_count += 1
        class_name = serialized.get("name", serialized.get("id", ["Unknown"])[-1])
        self._start_times["llm"] = time.time()
        
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[LLM开始] 模型: {class_name}")
        print(f"{indent}  提示词长度: {len(prompts[0]) if prompts else 0} 字符")
    
    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        """LLM 生成完成时调用"""
        elapsed = time.time() - self._start_times.pop("llm", time.time())
        token_usage = response.llm_output.get("token_usage", {}) if response.llm_output else {}
        
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[LLM完成] 耗时: {elapsed:.2f}s")
        print(f"{indent}  Token 使用: {token_usage}")
    
    def on_llm_error(self, error: Exception, **kwargs: Any) -> None:
        """LLM 出错时调用"""
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[LLM错误] {type(error).__name__}: {error}")
    
    # === Chain 回调 ===
    
    def on_chain_start(
        self,
        serialized: Dict[str, Any],
        inputs: Dict[str, Any],
        **kwargs: Any,
    ) -> None:
        """链开始执行时调用"""
        class_name = serialized.get("name", serialized.get("id", ["Chain"])[-1])
        self._chain_stack.append(class_name)
        self._start_times[f"chain_{class_name}"] = time.time()
        
        indent = "  " * (len(self._chain_stack) - 1)
        print(f"{indent}[链开始] {class_name}")
        print(f"{indent}  输入键: {list(inputs.keys())}")
    
    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> None:
        """链执行完成时调用"""
        class_name = self._chain_stack.pop() if self._chain_stack else "Unknown"
        elapsed = time.time() - self._start_times.pop(f"chain_{class_name}", time.time())
        
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[链完成] {class_name}, 耗时: {elapsed:.2f}s")
        output_keys = list(outputs.keys()) if isinstance(outputs, dict) else ["text"]
        print(f"{indent}  输出键: {output_keys}")
    
    # === Tool 回调 ===
    
    def on_tool_start(
        self,
        serialized: Dict[str, Any],
        input_str: str,
        **kwargs: Any,
    ) -> None:
        """工具开始执行时调用"""
        self.tool_call_count += 1
        tool_name = serialized.get("name", "UnknownTool")
        self._start_times[f"tool_{tool_name}"] = time.time()
        
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[工具开始] {tool_name}")
        print(f"{indent}  输入: {input_str[:100]}...")
    
    def on_tool_end(
        self,
        output: str,
        **kwargs: Any,
    ) -> None:
        """工具执行完成时调用"""
        elapsed = time.time() - self._start_times.pop(
            list(self._start_times.keys())[-1], time.time()
        )
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[工具完成] 耗时: {elapsed:.2f}s")
        print(f"{indent}  输出: {str(output)[:100]}...")
    
    def on_tool_error(self, error: Exception, **kwargs: Any) -> None:
        """工具执行出错时调用"""
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[工具错误] {type(error).__name__}: {error}")
    
    # === Retriever 回调 ===
    
    def on_retriever_start(
        self,
        serialized: Dict[str, Any],
        query: str,
        **kwargs: Any,
    ) -> None:
        """检索器开始检索时调用"""
        self.retriever_call_count += 1
        retriever_name = serialized.get("name", "UnknownRetriever")
        self._start_times["retriever"] = time.time()
        
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[检索开始] {retriever_name}")
        print(f"{indent}  查询: {query}")
    
    def on_retriever_end(
        self,
        documents: List[Any],
        **kwargs: Any,
    ) -> None:
        """检索器完成时调用"""
        elapsed = time.time() - self._start_times.pop("retriever", time.time())
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[检索完成] 耗时: {elapsed:.2f}s, 返回 {len(documents)} 个文档")
    
    # === 其他回调 ===
    
    def on_text(self, text: str, **kwargs: Any) -> None:
        """接收到文本块时调用（流式输出）"""
        pass  # 流式输出通常在 astream_events 中处理
    
    def on_agent_action(self, action, **kwargs: Any) -> None:
        """Agent 执行动作时调用"""
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[Agent动作] 工具: {action.tool}, 输入: {action.tool_input}")
    
    def on_agent_finish(self, finish, **kwargs: Any) -> None:
        """Agent 完成时调用"""
        indent = "  " * len(self._chain_stack)
        print(f"{indent}[Agent完成] 输出: {str(finish.return_values)[:100]}...")


# 测试回调处理器
print("=== 自定义回调处理器测试 ===\n")

debug_handler = DebugLoggingCallback()

print("回调处理器方法一览:")
for attr in dir(debug_handler):
    if attr.startswith("on_") and callable(getattr(debug_handler, attr)):
        print(f"  - {attr}")

print("\n# 使用方式1: 构造时传入")
print("chain = prompt | llm | output_parser")
print("chain.invoke(input, config={'callbacks': [DebugLoggingCallback()]})")

print("\n# 使用方式2: 全局设置")
print("from langchain_core.callbacks import get_callback_manager")
print("get_callback_manager().add_handler(DebugLoggingCallback())")

print("\n# 使用方式3: with_config()")
print("chain_with_logging = chain.with_config(")
print("    callbacks=[DebugLoggingCallback()]")
print(")")


## 5. Token 计数回调

实现一个追踪每次 LLM 调用 token 消耗，并累积总计的回调处理器。

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from typing import Any, Dict
from collections import defaultdict


class TokenCountingCallback(BaseCallbackHandler):
    """
    Token 计数回调处理器。
    
    记录每次 LLM 调用的 token 消耗:
    - prompt_tokens: 输入 token 数
    - completion_tokens: 输出 token 数
    - total_tokens: 总计
    """
    
    def __init__(self):
        super().__init__()
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_tokens = 0
        self.call_count = 0
        self.call_records: list = []
    
    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        """LLM 调用完成时累计 token"""
        self.call_count += 1
        
        # 从 response 中提取 token 使用量
        token_usage = {}
        if response.llm_output and "token_usage" in response.llm_output:
            token_usage = response.llm_output["token_usage"]
        
        # 也尝试从 generations 中获取（某些模型放在这里）
        if not token_usage and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    if hasattr(gen, 'generation_info'):
                        info = gen.generation_info or {}
                        if 'usage_metadata' in info:
                            token_usage = info['usage_metadata']
        
        prompt_tokens = token_usage.get("prompt_tokens", 0)
        completion_tokens = token_usage.get("completion_tokens", 0)
        total = token_usage.get("total_tokens", prompt_tokens + completion_tokens)
        
        self.total_prompt_tokens += prompt_tokens
        self.total_completion_tokens += completion_tokens
        self.total_tokens += total
        
        self.call_records.append({
            "call": self.call_count,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total,
        })
    
    def get_statistics(self) -> Dict[str, Any]:
        """获取统计摘要"""
        return {
            "总调用次数": self.call_count,
            "总输入 token": self.total_prompt_tokens,
            "总输出 token": self.total_completion_tokens,
            "总 token": self.total_tokens,
            "平均每次输入": self.total_prompt_tokens / max(self.call_count, 1),
            "平均每次输出": self.total_completion_tokens / max(self.call_count, 1),
            "调用记录": self.call_records,
        }
    
    def reset(self):
        """重置计数器"""
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_tokens = 0
        self.call_count = 0
        self.call_records = []


print("=== Token 计数回调演示 ===\n")

# 模拟使用
token_counter = TokenCountingCallback()

print("Token 计数回调功能:")
print("  - on_llm_end(): 自动从 LLMResult 提取 token 使用量")
print("  - get_statistics(): 获取统计摘要")
print("  - reset(): 重置计数器")
print()

print("# 使用示例:")
print("token_counter = TokenCountingCallback()")
print("chain.invoke(input, config={'callbacks': [token_counter]})")
print("stats = token_counter.get_statistics()")
print("print(f'总token: {stats[\"总 token\"]}')")

print("\n# 模拟统计输出:")
# 模拟一些数据
token_counter.total_prompt_tokens = 1520
token_counter.total_completion_tokens = 845
token_counter.total_tokens = 2365
token_counter.call_count = 5
print(token_counter.get_statistics())

## 6. 成本追踪回调

基于 token 使用量和模型定价，实时计算 API 调用成本。

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from typing import Any, Dict, Optional


class CostTrackingCallback(BaseCallbackHandler):
    """
    成本追踪回调处理器。
    
    根据模型定价表计算 API 调用费用:
    - gpt-4: $0.03/1K prompt tokens, $0.06/1K completion tokens
    - gpt-4-turbo: $0.01/1K prompt, $0.03/1K completion
    - gpt-3.5-turbo: $0.0005/1K prompt, $0.0015/1K completion
    
    注意: 价格会变动，请以官方最新定价为准。
    """
    
    # 每 1K token 的价格（美元）
    PRICING = {
        "gpt-4": {"prompt": 0.03, "completion": 0.06},
        "gpt-4-turbo": {"prompt": 0.01, "completion": 0.03},
        "gpt-4o": {"prompt": 0.005, "completion": 0.015},
        "gpt-4o-mini": {"prompt": 0.00015, "completion": 0.0006},
        "gpt-3.5-turbo": {"prompt": 0.0005, "completion": 0.0015},
    }
    
    def __init__(self, model_name: Optional[str] = None):
        super().__init__()
        self.model_name = model_name
        self.total_cost_usd = 0.0
        self.call_count = 0
        self.cost_history: list = []
    
    def on_llm_start(
        self,
        serialized: Dict[str, Any],
        prompts: list,
        **kwargs: Any,
    ) -> None:
        """记录使用的模型"""
        if not self.model_name:
            self.model_name = serialized.get("name", serialized.get("id", ["unknown"])[-1])
    
    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        """计算每次调用的成本"""
        self.call_count += 1
        
        # 尝试获取 token 使用量
        token_usage = {}
        if response.llm_output and "token_usage" in response.llm_output:
            token_usage = response.llm_output["token_usage"]
        
        prompt_tokens = token_usage.get("prompt_tokens", 0)
        completion_tokens = token_usage.get("completion_tokens", 0)
        
        # 查找定价
        pricing = self.PRICING.get(self.model_name, {})
        prompt_price = pricing.get("prompt", 0.001)  # 默认值
        completion_price = pricing.get("completion", 0.002)
        
        # 计算成本
        prompt_cost = (prompt_tokens / 1000) * prompt_price
        completion_cost = (completion_tokens / 1000) * completion_price
        call_cost = prompt_cost + completion_cost
        
        self.total_cost_usd += call_cost
        
        self.cost_history.append({
            "call": self.call_count,
            "model": self.model_name,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "cost_usd": round(call_cost, 6),
        })
    
    def get_summary(self) -> Dict[str, Any]:
        """获取成本摘要"""
        return {
            "模型": self.model_name,
            "调用次数": self.call_count,
            "总成本(USD)": round(self.total_cost_usd, 6),
            "总成本(CNY, 汇率7.25)": round(self.total_cost_usd * 7.25, 4),
            "成本明细": self.cost_history,
        }
    
    def reset(self):
        """重置追踪器"""
        self.total_cost_usd = 0.0
        self.call_count = 0
        self.cost_history = []


print("=== 成本追踪回调演示 ===\n")

cost_tracker = CostTrackingCallback(model_name="gpt-4o")

print("模型定价表 (每1K tokens, USD):")
for model, prices in CostTrackingCallback.PRICING.items():
    print(f"  {model:<16} 输入: ${prices['prompt']:<10} 输出: ${prices['completion']:<10}")

print()
print("# 使用示例:")
print("cost_tracker = CostTrackingCallback(model_name='gpt-4o')")
print("chain.invoke(input, config={'callbacks': [cost_tracker]})")
print("summary = cost_tracker.get_summary()")
print("print(f'本次会话成本: ${summary[\"总成本(USD)\"]:.4f}')")

print("\n# 模拟成本计算:")
# 模拟一些调用
cost_tracker.total_cost_usd = 0.0
cost_tracker.call_count = 0

print("\n使用 on_llm_end 模拟不同的调用:")

# 模拟 gpt-4o 的使用
print("  问题1 (简单): 100 prompt tokens, 50 completion tokens")
p_cost = (100 / 1000) * 0.005
c_cost = (50 / 1000) * 0.015
print(f"    成本: ${p_cost:.6f} + ${c_cost:.6f} = ${p_cost + c_cost:.6f}")

print("  问题2 (中等): 500 prompt tokens, 200 completion tokens")
p_cost2 = (500 / 1000) * 0.005
c_cost2 = (200 / 1000) * 0.015
print(f"    成本: ${p_cost2:.6f} + ${c_cost2:.6f} = ${p_cost2 + c_cost2:.6f}")

print("  问题3 (复杂): 2000 prompt tokens, 800 completion tokens")
p_cost3 = (2000 / 1000) * 0.005
c_cost3 = (800 / 1000) * 0.015
print(f"    成本: ${p_cost3:.6f} + ${c_cost3:.6f} = ${p_cost3 + c_cost3:.6f}")

total_sim = p_cost + c_cost + p_cost2 + c_cost2 + p_cost3 + c_cost3
print(f"\n  总计: ${total_sim:.6f} USD")
print(f"  约合: ¥{total_sim * 7.25:.4f} CNY")

print("\n# 成本优化建议:")
print("# 1. 简单任务使用 gpt-4o-mini (成本仅为 gpt-4o 的 ~3%)")
print("# 2. 使用缓存减少重复调用")
print("# 3. 合理设置 max_tokens 限制输出长度")
print("# 4. 定期使用 get_summary() 监控累计成本")
print("# 5. 设置每日成本上限，达到时停止调用")

## 7. LangSmith 追踪配置

LangSmith 是 LangChain 官方的调试、测试和监控平台。
通过环境变量配置，自动追踪所有链的执行。

In [ ]:
print("=== LangSmith 追踪配置 ===\n")

print("# LangSmith 是一个开发平台，提供:")
print("# - 执行追踪: 可视化链的每一步")
print("# - 调试工具: 查看中间状态")
print("# - 测试套件: 批量评估")
print("# - 监控面板: Token 使用和延迟")
print()

print("# 环境变量配置（在代码中设置）:")
print("""
import os

# 启用 LangSmith 追踪
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "ls__your-api-key-here"  # 从 https://smith.langchain.com 获取
os.environ["LANGCHAIN_PROJECT"] = "phase-05-lcel"  # 项目名称（可选，自动创建）
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"  # 默认即可
""")

print("\n# 不设置环境变量时的本地内联追踪:")
print("""
# 即使没有 LangSmith API Key，仍可使用回调进行本地追踪
# 所有链的执行都会被回调处理器捕获

from langchain_core.tracers import ConsoleCallbackHandler

# 控制台追踪 - 打印到 stdout
chain.invoke(
    input,
    config={"callbacks": [ConsoleCallbackHandler()]}
)
""")

print("\n# 为特定链添加标签用于过滤:")
print("""
# 在 LangSmith 仪表板中按标签筛选追踪
chain.invoke(
    input,
    config={
        "tags": ["production", "phase-05"],
        "metadata": {
            "user_id": "user_123",
            "session_id": "session_abc",
            "feature": "rag-qa",
        }
    }
)
""")

print("\n# LangSmith 配置检查清单:")
print("# [ ] 注册 LangSmith 账号: https://smith.langchain.com")
print("# [ ] 获取 API Key: Settings -> API Keys")
print("# [ ] 设置环境变量 LANGCHAIN_TRACING_V2=true")
print("# [ ] 设置环境变量 LANGCHAIN_API_KEY")
print("# [ ] 可选: 设置 LANGCHAIN_PROJECT 组织追踪")
print("# [ ] 验证: 运行一个链后在 LangSmith UI 查看")

print("\n# 环境变量设置方式:")
print("# 方式1: 终端 export LANGCHAIN_API_KEY=ls__...")
print("# 方式2: .env 文件 + python-dotenv")
print("# 方式3: 代码中 os.environ['...'] = '...'（不推荐提交到 git）")
print("# 方式4: 部署平台的环境变量配置（推荐生产环境）")

## 8. 综合示例 - 带回调的 RAG 链

将流式处理、Token 计数、成本追踪和调试日志组合在一个完整的链中。

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

print("=== 综合示例: 带监控的 RAG 链 ===\n")

print("# 构建完整的链，并附加多个回调:")
print("""
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 创建回调处理器
debug_handler = DebugLoggingCallback()
token_counter = TokenCountingCallback()
cost_tracker = CostTrackingCallback(model_name='gpt-4o')

# 2. 构建 RAG 链
prompt = ChatPromptTemplate.from_messages([
    ("system", "基于以下上下文回答问题。如果上下文中没有，请说不知道。\\n\\n上下文: {context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    '''格式化检索到的文档'''
    return "\\n\\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 3. 使用 with_config 固定回调
monitored_chain = rag_chain.with_config(
    callbacks=[debug_handler, token_counter, cost_tracker],
    tags=["rag-production"],
    metadata={"version": "1.0.0"},
)

# 4. 执行查询
try:
    result = monitored_chain.invoke("什么是RAG？")
    print(f"回答: {result}")
except Exception as e:
    print(f"执行错误: {e}")

# 5. 获取统计
print(f"\\n=== 统计信息 ===")
print(f"Token 统计: {token_counter.get_statistics()}")
print(f"成本统计: {cost_tracker.get_summary()}")
""")

print("\n# 多个回调的组合策略:")
print("#   DebugLoggingCallback  -> 开发/调试环境")
print("#   TokenCountingCallback  -> 开发/测试环境")
print("#   CostTrackingCallback   -> 生产环境")
print("#   LangSmith               -> 全生命周期")
print("#   ConsoleCallbackHandler  -> 快速调试")
print()
print("# 环境推荐配置:")
print("#   开发: [DebugLoggingCallback, TokenCountingCallback]")
print("#   测试: [TokenCountingCallback, CostTrackingCallback]")
print("#   生产: [CostTrackingCallback, LangSmith]")
print("#   CI/CD: [TokenCountingCallback]")

print("\n# 使用 with_config 的好处:")
print("# 1. 不会修改原链，保持链的纯净性")
print("# 2. 可以为不同环境创建不同的配置")
print("# 3. 标签和元数据用于 LangSmith 过滤和组织")
print("# 4. 回调配置会被传递给链中的所有子组件")

## 总结

| 功能 | 方法/类 | 用途 |
|------|---------|------|
| 同步流式 | .stream() | 逐 token 输出 |
| 异步流式 | .astream() | 异步逐 token 输出 |
| 事件流式 | .astream_events() | 细粒度事件监控 |
| 回调基类 | BaseCallbackHandler | 自定义监控逻辑 |
| Token 计数 | 自定义回调 | 统计 token 使用 |
| 成本追踪 | 自定义回调 | 计算 API 费用 |
| LangSmith | 环境变量配置 | 云端追踪和调试 |
| 配置传递 | config={'callbacks': [...]} | 运行时设置回调 |

回调方法覆盖表：

| 回调方法 | 触发时机 | 参数 |
|----------|----------|------|
| on_llm_start | LLM 开始生成 | serialized, prompts |
| on_llm_end | LLM 生成完成 | response (LLMResult) |
| on_llm_error | LLM 出错 | error |
| on_chain_start | 链开始执行 | serialized, inputs |
| on_chain_end | 链执行完成 | outputs |
| on_tool_start | 工具开始执行 | serialized, input_str |
| on_tool_end | 工具执行完成 | output |
| on_tool_error | 工具出错 | error |
| on_retriever_start | 检索开始 | serialized, query |
| on_retriever_end | 检索完成 | documents |
| on_agent_action | Agent 执行动作 | action |
| on_agent_finish | Agent 完成 | finish |